In [2]:
import numpy as np
import pandas as pd
import neurokit2 as nk

from pathlib import Path
from time import perf_counter

FS = 250
OUTPUT_DIR = Path("r_peak_database")

In [3]:
def calculate_hrv_windows(all_peaks, quality_runs, sampling_rate=250):
    group_columns = [
        "Subject_ID", "Recording_ID", "Night_ID", "Quality_Run_ID"
    ]
    window_samples = 300 * sampling_rate
    rows = []

    peak_groups = {
        key: group.sort_values("Sample_In_Run")
        for key, group in all_peaks.groupby(group_columns, sort=False)
    }

    for _, run in quality_runs.iterrows():
        key = tuple(run[column] for column in group_columns)
        run_peaks = peak_groups.get(key, all_peaks.iloc[:0])

        # Only use complete five-minute windows.
        window_count = int(run["Samples"]) // window_samples

        for window_number in range(window_count):
            start = window_number * window_samples
            stop = start + window_samples

            window = run_peaks.loc[
                run_peaks["Sample_In_Run"].ge(start)
                & run_peaks["Sample_In_Run"].lt(stop)
            ].copy()

            # The first peak's RR interval starts outside this window.
            # Count flags only for intervals between peaks in the window.
            flagged_count = int(
                window["Needs_Review"].iloc[1:].sum()
            )

            row = {
                **dict(zip(group_columns, key)),
                "Window_Number": window_number,
                "Window_Start": (
                    pd.Timestamp(run["Start_Time"])
                    + pd.Timedelta(seconds=start / sampling_rate)
                ),
                "Window_End": (
                    pd.Timestamp(run["Start_Time"])
                    + pd.Timedelta(seconds=stop / sampling_rate)
                ),
                "Peak_Count": len(window),
                "Flagged_Interval_Count": flagged_count,
                "Needs_Review": flagged_count > 0,
                "RMSSD_ms": np.nan,
                "LF_HF_Ratio": np.nan,
                "HRV_Status": "Insufficient data",
                "Error_Message": ""
            }

            if len(window) >= 3:
                # Positions relative to the beginning of this window.
                positions = (
                    window["Sample_In_Run"].to_numpy(dtype=int) - start
                )

                try:
                    time_metrics = nk.hrv_time(
                        positions,
                        sampling_rate=sampling_rate,
                        show=False
                    )

                    row["RMSSD_ms"] = float(
                        time_metrics["HRV_RMSSD"].iloc[0]
                    )

                    frequency_metrics = nk.hrv_frequency(
                        positions,
                        sampling_rate=sampling_rate,
                        psd_method="welch",
                        show=False
                    )

                    row["LF_HF_Ratio"] = float(
                        frequency_metrics["HRV_LFHF"].iloc[0]
                    )

                    if (
                        np.isfinite(row["RMSSD_ms"])
                        and np.isfinite(row["LF_HF_Ratio"])
                    ):
                        row["HRV_Status"] = "Calculated"
                    else:
                        row["HRV_Status"] = "Calculation failed"
                        row["Error_Message"] = (
                            "One or both HRV metrics are not finite."
                        )

                except Exception as error:
                    row["HRV_Status"] = "Calculation failed"
                    row["Error_Message"] = (
                        f"{type(error).__name__}: {error}"
                    )

            rows.append(row)

    columns = group_columns + [
        "Window_Number", "Window_Start", "Window_End",
        "Peak_Count", "Flagged_Interval_Count", "Needs_Review",
        "RMSSD_ms", "LF_HF_Ratio", "HRV_Status", "Error_Message"
    ]

    return pd.DataFrame(rows, columns=columns)

In [4]:
processing_log = pd.read_csv(OUTPUT_DIR / "processing_log.csv")

completed = processing_log.loc[
    processing_log["Status"].eq("Completed")
]

if completed.empty:
    raise ValueError("No completed recordings available.")

all_peaks = pd.read_parquet(OUTPUT_DIR / "all_r_peaks.parquet")

quality_runs = pd.concat(
    [pd.read_parquet(path) for path in completed["Runs_Path"]],
    ignore_index=True
)

hrv_windows = calculate_hrv_windows(all_peaks, quality_runs, sampling_rate=FS)

hrv_windows.to_parquet(
    OUTPUT_DIR / "hrv_windows.parquet",
    index=False
)

display(hrv_windows.head())
display(hrv_windows["HRV_Status"].value_counts())

print("Windows needing review:", int(hrv_windows["Needs_Review"].sum()))

,Subject_ID,Recording_ID,Night_ID,Quality_Run_ID,Window_Number,Window_Start,Window_End,Peak_Count,Flagged_Interval_Count,Needs_Review,RMSSD_ms,LF_HF_Ratio,HRV_Status,Error_Message
0,c1s01,zephyr/2022_06_09-22_02_32,2022-06-09,0,0,2022-06-10 03:40:18.950,2022-06-10 03:45:18.950,283,2,True,74.953651,1.597282,Calculated,
1,c1s01,zephyr/2022_06_09-22_02_32,2022-06-09,0,1,2022-06-10 03:45:18.950,2022-06-10 03:50:18.950,279,9,True,90.617775,2.773636,Calculated,
2,c1s01,zephyr/2022_06_09-22_02_32,2022-06-09,2,0,2022-06-10 03:54:14.950,2022-06-10 03:59:14.950,287,3,True,75.306904,1.146270,Calculated,
3,c1s01,zephyr/2022_06_09-22_02_32,2022-06-09,4,0,2022-06-10 04:05:00.950,2022-06-10 04:10:00.950,291,9,True,89.353851,3.178014,Calculated,
4,c1s01,zephyr/2022_06_09-22_02_32,2022-06-09,4,1,2022-06-10 04:10:00.950,2022-06-10 04:15:00.950,283,0,False,35.354433,0.612911,Calculated,


HRV_Status
Calculated    10399
Name: count, dtype: int64

Windows needing review: 5163
